# Pré-processamento 8 — Revisão de compatibilidade entre palavra e categoria
## Decisões aprovadas
Aplicar 35 correções do grupo Nome Comun Masculino Singular e 24 do grupo Nome Comun Feminino Singular (59 aprovadas ao todo). Não alterar LEMA.
O código aceita categorias já corrigidas, permitindo reexecução. Valida palavras pela identidade e pelo lema, não pela antiga posição da linha. As linhas históricas abaixo se referem à versão anterior.
As análises linguísticas são revisão humana assistida; o script reproduz decisões explícitas, não um classificador automático.
Para acrescentar remoções, edite a lista e reexecute desde o carregamento. Uma cópia de segurança e um histórico de mudanças ficam em outputs/pre_processamento_8.
Somente backups e historico.json ficam na pasta de apoio. Arquivos intermediários são criados em pasta temporária e limpos automaticamente ao concluir ou ocorrer uma exceção.

In [9]:
from pathlib import Path
import json, hashlib, shutil, os, subprocess
import pandas as pd
from IPython.display import display
arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
apoio = Path("outputs/pre_processamento_8")
apoio.mkdir(parents=True, exist_ok=True)
base = pd.read_excel(arquivo)
base_original = base.copy(deep=True)
hash_entrada = hashlib.sha256(arquivo.read_bytes()).hexdigest()
print("Registros carregados:", len(base))


Registros carregados: 7812


## 1. Função de correção de categoria e preparação do resultado

In [10]:
# Função compartilhada pelos grupos; todo o código de correção está neste notebook.
# Altera exclusivamente CATMORF, sem atribuir valores em LEMA.
def aplicar_correcoes_categoria(resultado, correcoes):
    antes = resultado.copy(deep=True)
    aplicadas = []
    for item in correcoes:
        mascara = resultado["PALAVRA"].eq(item["PALAVRA"])
        assert mascara.sum() == 1, f"Palavra ausente ou duplicada: {item['PALAVRA']}"
        indice = resultado.index[mascara][0]
        atual = resultado.at[indice, "CATMORF"]
        assert atual in (item["CATMORF"], item["CATEGORIA_SUGERIDA"]), f"Categoria mudou: {item['PALAVRA']}"
        assert resultado.at[indice, "LEMA"] == item["LEMA"], f"Lema mudou: {item['PALAVRA']}"
        if atual != item["CATEGORIA_SUGERIDA"]:
            resultado.at[indice, "CATMORF"] = item["CATEGORIA_SUGERIDA"]
            aplicadas.append({**item, "CATEGORIA_ANTERIOR": atual})
    pd.testing.assert_frame_equal(antes.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
    return aplicadas

resultado = base.copy(deep=True)
alteracoes = []
correcoes_aprovadas = []


## 1.1. Nome Comun Masculino Singular — aplicar 35 correções aprovadas

In [11]:
# Lista completa das decisões aprovadas — Nome Comun Masculino Singular
correcoes_masculino = [{'LINHA_EXCEL': 237,
  'PALAVRA': 'ajudando',
  'LEMA': 'ajudá-lo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de ajudar; não corresponde a substantivo masculino singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 246,
  'PALAVRA': 'ajudá-lo',
  'LEMA': 'ajudador',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo verbal com pronome enclítico; classificar o componente verbal, '
                   'preservando a palavra completa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 253,
  'PALAVRA': 'albuns',
  'LEMA': 'alazão',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Forma plural de álbum, sem acento no dado original; propõe-se apenas corrigir '
                   'o número, preservando a grafia.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 258,
  'PALAVRA': 'alcaparra',
  'LEMA': 'alcanço',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 261,
  'PALAVRA': 'aldeia',
  'LEMA': 'alce',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 269,
  'PALAVRA': 'alergia',
  'LEMA': 'alemão',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 274,
  'PALAVRA': 'algo',
  'LEMA': 'alfinete',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Pronome Indefinido Invariável Indefinido',
  'JUSTIFICATIVA': 'Indefinido invariável, também empregado como advérbio; a proposta pronominal é '
                   'uma análise possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 276,
  'PALAVRA': 'algum',
  'LEMA': 'algodão',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Determinante Indefinido Masculino Singular Indefinido',
  'JUSTIFICATIVA': 'Indefinido masculino singular; pode funcionar como determinante ou pronome, '
                   'não como nome comum nessa análise lexical.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 291,
  'PALAVRA': 'almofadas',
  'LEMA': 'almoço',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Substantivo feminino no plural; gênero e número incompatíveis com masculino '
                   'singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 292,
  'PALAVRA': 'almoça',
  'LEMA': 'alívio',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de almoçar; também admite imperativo. Presente do indicativo é uma opção '
                   'válida, sem preferência entre análises verbais possíveis.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 304,
  'PALAVRA': 'aluga',
  'LEMA': 'alto',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de alugar; também admite imperativo. A proposta é uma análise verbal '
                   'possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 309,
  'PALAVRA': 'alunos',
  'LEMA': 'aluguel',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Forma plural do substantivo; a classificação singular não corresponde à '
                   'palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 351,
  'PALAVRA': 'ameaçou',
  'LEMA': 'ambiente',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de ameaçar no pretérito perfeito, terceira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 357,
  'PALAVRA': 'americanas',
  'LEMA': 'amendoim',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Admite leitura como substantivo, mas na forma feminina plural; preservar a '
                   'classe nominal e corrigir gênero/número.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 361,
  'PALAVRA': 'amigas',
  'LEMA': 'americano',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Admite leitura como substantivo, mas na forma feminina plural; preservar a '
                   'classe nominal e corrigir gênero/número.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 364,
  'PALAVRA': 'amizade',
  'LEMA': 'amigo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 369,
  'PALAVRA': 'amostra',
  'LEMA': 'amo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 370,
  'PALAVRA': 'amostras',
  'LEMA': 'amor',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Substantivo feminino no plural; gênero e número incompatíveis com masculino '
                   'singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 379,
  'PALAVRA': 'anda',
  'LEMA': 'anão',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de andar ou, como substantivo, feminino (anda/perna de pau). Masculino '
                   'singular não se sustenta nessas leituras; a proposta verbal é possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 402,
  'PALAVRA': 'anoitecendo',
  'LEMA': 'anjo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de anoitecer; não corresponde a substantivo masculino singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 403,
  'PALAVRA': 'anos',
  'LEMA': 'ano',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Forma plural do substantivo; a classificação singular não corresponde à '
                   'palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 416,
  'PALAVRA': 'antigas',
  'LEMA': 'antídoto',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Plural',
  'JUSTIFICATIVA': 'Forma feminina plural de antigo. Mesmo em leitura substantivada, masculino '
                   'singular é incompatível; adjetivo é uma proposta possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2247,
  'PALAVRA': 'der',
  'LEMA': 'der',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Futuro Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma do futuro do subjuntivo de dar; primeira e terceira pessoas do singular '
                   'são possíveis. Propõe-se uma delas sem preferência.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2251,
  'PALAVRA': 'derrete',
  'LEMA': 'derrete',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de derreter; também admite imperativo. A proposta é uma análise verbal '
                   'possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2463,
  'PALAVRA': 'dica',
  'LEMA': 'dica',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2483,
  'PALAVRA': 'dinamite',
  'LEMA': 'dinamite',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2538,
  'PALAVRA': 'devagarzinho',
  'LEMA': 'devagarzinho',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Advérbio Geral',
  'JUSTIFICATIVA': 'Forma diminutiva de devagar, com valor adverbial de modo.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2662,
  'PALAVRA': 'dói',
  'LEMA': 'dói',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de doer no presente, terceira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2759,
  'PALAVRA': 'encaixo',
  'LEMA': 'encaixo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de encaixar no presente, primeira pessoa do singular; o substantivo '
                   'usual é encaixe.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2831,
  'PALAVRA': 'enganá-lo',
  'LEMA': 'enganá-lo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo verbal com pronome enclítico; classificar o componente verbal, '
                   'preservando a palavra completa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4245,
  'PALAVRA': 'jogue',
  'LEMA': 'jogue',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de jogar no presente do subjuntivo ou imperativo; propõe-se uma análise '
                   'verbal possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4318,
  'PALAVRA': 'lanchonete',
  'LEMA': 'lanchonete',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4408,
  'PALAVRA': 'levanto',
  'LEMA': 'levanto',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de levantar no presente, primeira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6667,
  'PALAVRA': 'salvá-lo',
  'LEMA': 'salvá-lo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo verbal com pronome enclítico; classificar o componente verbal, '
                   'preservando a palavra completa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7421,
  'PALAVRA': 'troque',
  'LEMA': 'troque',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de trocar no presente do subjuntivo ou imperativo; propõe-se uma análise '
                   'verbal possível.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_masculino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_masculino)
print("Grupo: Nome Comun Masculino Singular")
print("Correções aprovadas:", len(correcoes_masculino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_masculino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_masculino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Masculino Singular
Correções aprovadas: 35
Aplicadas nesta execução: 0
Já estavam corrigidas: 35


,PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
0,ajudando,ajudá-lo,Nome Comun Masculino Singular,Verbo Principal Gerúndio,Aprovada
1,ajudá-lo,ajudador,Nome Comun Masculino Singular,Verbo Principal Infinitivo,Aprovada
2,albuns,alazão,Nome Comun Masculino Singular,Nome Comun Masculino Plural,Aprovada
3,alcaparra,alcanço,Nome Comun Masculino Singular,Nome Comun Feminino Singular,Aprovada
4,aldeia,alce,Nome Comun Masculino Singular,Nome Comun Feminino Singular,Aprovada
5,alergia,alemão,Nome Comun Masculino Singular,Nome Comun Feminino Singular,Aprovada
6,algo,alfinete,Nome Comun Masculino Singular,Pronome Indefinido Invariável Indefinido,Aprovada
7,algum,algodão,Nome Comun Masculino Singular,Determinante Indefinido Masculino Singular Ind...,Aprovada
8,almofadas,almoço,Nome Comun Masculino Singular,Nome Comun Feminino Plural,Aprovada
9,almoça,alívio,Nome Comun Masculino Singular,Verbo Principal Indicativo Presente Terceira P...,Aprovada


## 1.2. Nome Comun Feminino Singular — aplicar 24 correções aprovadas

In [12]:
# Lista completa das decisões aprovadas — Nome Comun Feminino Singular
correcoes_feminino = [{'LINHA_EXCEL': 235,
  'PALAVRA': 'ajudador',
  'LEMA': 'ajuda',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'A forma ajudador é masculina; ajudadora é a correspondente feminina. Manter a '
                   'leitura nominal e corrigir o gênero.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 247,
  'PALAVRA': 'ajudá-los',
  'LEMA': 'ajudadora',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de ajudar com pronome enclítico; não corresponde a nome comum '
                   'feminino.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 259,
  'PALAVRA': 'alce',
  'LEMA': 'alcaparra',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O substantivo que designa o animal é masculino; também há leitura verbal, mas '
                   'não é necessário preferi-la.',
  'REFERENCIA': 'https://dicionario.priberam.org/alce',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 262,
  'PALAVRA': 'alegrar',
  'LEMA': 'aldeia',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma infinitiva verbal. Uma eventual leitura nominal do infinitivo seria '
                   'masculina; a categoria feminina atual não é compatível.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 271,
  'PALAVRA': 'alfabeto',
  'LEMA': 'alergia',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; feminino singular não corresponde à forma '
                   'lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 294,
  'PALAVRA': 'almoçar',
  'LEMA': 'almeida',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma infinitiva verbal. Uma eventual leitura nominal do infinitivo seria '
                   'masculina; a categoria feminina atual não é compatível.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 306,
  'PALAVRA': 'aluguel',
  'LEMA': 'altura',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; feminino singular não corresponde à forma '
                   'lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 313,
  'PALAVRA': 'além',
  'LEMA': 'alegria',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Além admite substantivo masculino (o além) e advérbio; não há motivo para '
                   'preferir uma classe, então manter a nominal com gênero compatível.',
  'REFERENCIA': 'https://dicionario.priberam.org/além',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 333,
  'PALAVRA': 'amarrado',
  'LEMA': 'amarra',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino (volume ou conjunto atado), além de '
                   'adjetivo/particípio. Preservar a leitura nominal possível e corrigir o gênero.',
  'REFERENCIA': 'https://dicionario.priberam.org/amarrado',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 334,
  'PALAVRA': 'amarrar',
  'LEMA': 'amarra',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma infinitiva verbal. Uma eventual leitura nominal do infinitivo seria '
                   'masculina; a categoria feminina atual não é compatível.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 356,
  'PALAVRA': 'amendoins',
  'LEMA': 'amelinha',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Plural masculino de amendoim; a categoria atual erra gênero e número.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 362,
  'PALAVRA': 'amigo',
  'LEMA': 'amígdala',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; feminino singular não corresponde à forma '
                   'lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 367,
  'PALAVRA': 'amor',
  'LEMA': 'amizade',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; feminino singular não corresponde à forma '
                   'lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 376,
  'PALAVRA': 'amígdalas',
  'LEMA': 'amigo',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Plural de amígdala; corrigir apenas o número.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 380,
  'PALAVRA': 'andando',
  'LEMA': 'âncora',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de andar; a categoria de nome comum feminino singular não corresponde '
                   'à forma.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 413,
  'PALAVRA': 'antes',
  'LEMA': 'antena',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Antes admite substantivo masculino (o antes e o depois), além de advérbio; '
                   'manter a leitura nominal com gênero compatível.',
  'REFERENCIA': 'https://dicionario.priberam.org/antes',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1662,
  'PALAVRA': 'cocoricó',
  'LEMA': 'cocoricó',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'A onomatopeia admite leitura nominal masculina para o canto do galo; preservar '
                   'essa classe e corrigir o gênero.',
  'REFERENCIA': 'https://dicionario.priberam.org/cocoricó',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2452,
  'PALAVRA': 'dezessete',
  'LEMA': 'dezessete',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Quantidade',
  'JUSTIFICATIVA': 'Numeral cardinal; Quantidade é a classificação usada na base para numerais. '
                   'Também admite uso nominal masculino, mas feminino singular não corresponde à '
                   'leitura lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2625,
  'PALAVRA': 'drible',
  'LEMA': 'drible',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O substantivo drible é masculino; também há leitura verbal, que não precisa '
                   'ser preferida.',
  'REFERENCIA': 'https://dicionario.priberam.org/drible',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3644,
  'PALAVRA': 'fubá',
  'LEMA': 'fubá',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura nominal, fubá é masculino.',
  'REFERENCIA': 'https://dicionario.priberam.org/fubá',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3885,
  'PALAVRA': 'guarda-chuva',
  'LEMA': 'guarda-chuva',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O composto é substantivo masculino; o gênero de chuva não determina o gênero '
                   'do composto.',
  'REFERENCIA': 'https://dicionario.priberam.org/guarda-chuva',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6210,
  'PALAVRA': 'quebra-cabeça',
  'LEMA': 'quebra-cabeça',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O composto é substantivo masculino; não acompanha o gênero de cabeça.',
  'REFERENCIA': 'https://dicionario.priberam.org/quebra-cabeça',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6653,
  'PALAVRA': 'samba',
  'LEMA': 'samba',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura nominal registrada para dança, música ou baile, samba é masculino; '
                   'preservar a classe nominal.',
  'REFERENCIA': 'https://dicionario.priberam.org/samba',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7796,
  'PALAVRA': 'ímã',
  'LEMA': 'ímã',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O substantivo que designa o objeto magnético é masculino.',
  'REFERENCIA': 'https://dicionario.priberam.org/ímã',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_feminino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_feminino)
print("Grupo: Nome Comun Feminino Singular")
print("Correções aprovadas:", len(correcoes_feminino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_feminino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_feminino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Feminino Singular
Correções aprovadas: 24
Aplicadas nesta execução: 0
Já estavam corrigidas: 24


,PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
0,ajudador,ajuda,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
1,ajudá-los,ajudadora,Nome Comun Feminino Singular,Verbo Principal Infinitivo,Aprovada
2,alce,alcaparra,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
3,alegrar,aldeia,Nome Comun Feminino Singular,Verbo Principal Infinitivo,Aprovada
4,alfabeto,alergia,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
5,almoçar,almeida,Nome Comun Feminino Singular,Verbo Principal Infinitivo,Aprovada
6,aluguel,altura,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
7,além,alegria,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
8,amarrado,amarra,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
9,amarrar,amarra,Nome Comun Feminino Singular,Verbo Principal Infinitivo,Aprovada


## 2. Palavras identificadas como fora do escopo

In [13]:
palavras_remover = ['atamedama', 'bibolo', 'cafeta', 'casma', 'cehgar', 'dexo', 'encontralo', 'esto', 'euxo', 'letre', 'moder', 'niclodaplo', 'nicrodoplo', 'ocabelo', 'opapaiurso', 'sarlso', 'siamem', 'noís', 'autie', 'bebeça', 'bibibiva', 'chemana', 'chocola', 'dade', 'dede', 'emelie', 'iolha', 'levantese', 'manamana', 'melie', 'migatina', 'mimie', 'moite', 'nacidade', 'nose', 'olele', 'parchiva', 'picala', 'puxavida', 'sentese', 'vendese']
total_antes = len(resultado)
mascara_remocao = resultado["PALAVRA"].isin(palavras_remover)
removidas = resultado.loc[mascara_remocao].copy()
nao_encontradas = sorted(set(palavras_remover) - set(removidas["PALAVRA"]))
resultado = resultado.loc[~mascara_remocao].reset_index(drop=True)
quantidade_removida = len(removidas)
assert total_antes - quantidade_removida == len(resultado)
print("Remoções previstas nesta execução:", quantidade_removida)
print("Total previsto na base:", len(resultado))
if nao_encontradas:
    print("Palavras já removidas ou não encontradas:", nao_encontradas)
display(removidas[["PALAVRA", "LEMA", "CATMORF"]])


Remoções previstas nesta execução: 23
Total previsto na base: 7789
Palavras já removidas ou não encontradas: ['atamedama', 'bibolo', 'cafeta', 'casma', 'cehgar', 'dexo', 'encontralo', 'esto', 'euxo', 'letre', 'moder', 'niclodaplo', 'nicrodoplo', 'noís', 'ocabelo', 'opapaiurso', 'sarlso', 'siamem']


,PALAVRA,LEMA,CATMORF
697,autie,autie,Nome Comun Feminino Singular
880,bebeça,bebeça,Nome Comun Feminino Singular
929,bibibiva,bibibiva,Nome Comun Feminino Singular
1530,chemana,chemana,Nome Comun Feminino Singular
1554,chocola,chocola,Nome Comun Feminino Singular
2135,dade,dade,Nome Comun Feminino Singular
2173,dede,dede,Nome Comun Feminino Singular
2715,emelie,emelie,Nome Comun Feminino Singular
4147,iolha,iolha,Nome Comun Feminino Singular
4395,levantese,levantese,Nome Comun Feminino Singular


## 3. Salvamento e contagens

In [14]:
assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada, "A fonte mudou durante a execução."
retidas = base_original.loc[~base_original["PALAVRA"].isin(palavras_remover)].reset_index(drop=True)
pd.testing.assert_frame_equal(retidas.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
backup = apoio / f"backup_{hash_entrada[:16]}.xlsx"
if not backup.exists():
    shutil.copy2(arquivo, backup)
from tempfile import TemporaryDirectory
# A limpeza inclui arquivos de inspeção gerados pela ferramenta, mesmo em caso de erro.
with TemporaryDirectory(prefix="temporarios_", dir=apoio) as pasta_temporaria:
    temporario = Path(pasta_temporaria) / "base_verificacao.xlsx"
    pacote = Path(pasta_temporaria) / "operacao.json"
    matriz = [list(resultado.columns)] + json.loads(resultado.to_json(orient="values", force_ascii=False))
    pacote.write_text(json.dumps({
        "source": str(arquivo.resolve()), "target": str(temporario.resolve()),
        "originalRows": len(base_original) + 1, "values": matriz
    }, ensure_ascii=False), encoding="utf-8")
    runtime = Path(os.environ.get("USERPROFILE", str(Path.home()))) / ".cache/codex-runtimes/codex-primary-runtime/dependencies/node"
    node = runtime / "bin/node.exe"
    ambiente = os.environ.copy()
    ambiente["CODEX_NODE_PACKAGES"] = str(runtime / "node_modules")
    codigo_js = "\nimport fs from 'node:fs/promises';\nimport {createRequire} from 'node:module';\nimport {pathToFileURL} from 'node:url';\nconst require = createRequire(process.env.CODEX_NODE_PACKAGES + '/../package.json');\nconst {FileBlob, SpreadsheetFile} = await import(pathToFileURL(require.resolve('@oai/artifact-tool')).href);\nconst job=JSON.parse(await fs.readFile(process.argv[1],'utf8'));\nconst wb=await SpreadsheetFile.importXlsx(await FileBlob.load(job.source));\nconst sheet=wb.worksheets.getItem('Sheet1');\nsheet.getRange('A1:H'+job.originalRows).clear({applyTo:'contents'});\nsheet.getRange('A1:H'+job.values.length).values=job.values;\nwb.recalculate();\nconsole.log((await wb.inspect({kind:'region',sheetId:'Sheet1',range:'A1:C5',maxChars:1000})).ndjson);\nconst out=await SpreadsheetFile.exportXlsx(wb);\nawait out.save(job.target);\n"
    processo = subprocess.run([str(node), "--input-type=module", "-e", codigo_js, str(pacote.resolve())],
                             env=ambiente, capture_output=True, text=True, encoding="utf-8")
    if processo.returncode:
        raise RuntimeError(processo.stderr or processo.stdout)
    verificada = pd.read_excel(temporario)
    pd.testing.assert_frame_equal(resultado, verificada, check_dtype=False)
    assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada
    if not resultado.equals(base_original):
        os.replace(temporario, arquivo)
# Histórico por hashes de entrada e saída: registrar novas decisões após execuções sem alterações.
historico_path = apoio / "historico.json"
historico = json.loads(historico_path.read_text(encoding="utf-8")) if historico_path.exists() else []
hash_saida_atual = hashlib.sha256(arquivo.read_bytes()).hexdigest()
if not any(h["hash_entrada"] == hash_entrada and h.get("hash_saida") == hash_saida_atual for h in historico):
    historico.append({
        "hash_entrada": hash_entrada, "hash_saida": hashlib.sha256(arquivo.read_bytes()).hexdigest(),
        "total_antes": total_antes, "correcoes": len(alteracoes),
        "remocoes": quantidade_removida, "total_depois": len(resultado),
        "palavras_removidas": removidas["PALAVRA"].tolist(),
        "alteracoes": alteracoes
    })
historico_path.write_text(json.dumps(historico, ensure_ascii=False, indent=2), encoding="utf-8")
print("Correções aplicadas nesta execução:", len(alteracoes))
print("Remoções nesta execução:", quantidade_removida)
print("Remoções acumuladas pelo arquivo 8:", sum(h["remocoes"] for h in historico))
print("Total atual na base:", len(verificada))
print("Lemas e demais colunas das linhas mantidas: preservados.")


Correções aplicadas nesta execução: 0
Remoções nesta execução: 23
Remoções acumuladas pelo arquivo 8: 41
Total atual na base: 7789
Lemas e demais colunas das linhas mantidas: preservados.
